---
title: Tracks widget
---

In [1]:
import numpy as np
import pandas as pd
from geneinfo.widget import Tracks
from vscodenb import set_vscode_theme
set_vscode_theme()

In [2]:
#| echo: false
import os
if 'NBCONVERT_BGCOLOR' in os.environ:
    bg_color = os.environ['NBCONVERT_BGCOLOR']
    from geneinfo.widget import get_default_theme, set_default_theme
    theme = get_default_theme()
    theme['bg'] = bg_color
    theme['panel'] = bg_color
    theme['border'] = bg_color
    set_default_theme(theme)

Some synthetic data:

In [3]:
#| echo: false
from geneinfo.coords import chromosome_lengths
chrom_sizes = dict(chromosome_lengths(assembly='hg38'))
rng = np.random.default_rng(42)
quant_rows = []
n_points = 1000
for chrom, csz in chrom_sizes.items():
    positions = np.sort(rng.integers(0, csz, n_points))
    fst = np.abs(rng.normal(0.05, 0.08, n_points)).clip(0, 1)
    rate = np.cumsum(rng.normal(0, 0.3, n_points))
    rate = (rate - rate.min()) / (rate.max() - rate.min()) * 5 + 0.5
    mid = np.cumsum(rng.normal(0, 0.15, n_points))
    eig = np.sin(positions / 1e7 * 2 * np.pi) + rng.normal(0, 0.2, n_points)
    ci_lo = mid - rng.exponential(0.3, n_points)
    ci_hi = mid + rng.exponential(0.3, n_points) + 10
    depth = rng.poisson(30, n_points).astype(float)

    for pop in ['AFR', 'EUR']:
        noise = rng.normal(0, 0.02, n_points)
        for i in range(n_points):
            quant_rows.append({
                'chrom': chrom, 'pos': int(positions[i]),
                'fst': float(fst[i] + noise[i]),
                'rate': float(rate[i] + noise[i] * 10) + (5 if pop == 'EUR' else 0),
                'eig': float(eig[i] + noise[i]),
                'ci_lo': float(ci_lo[i]) + (15 if pop == 'EUR' else 0), 
                'ci_hi': float(ci_hi[i]) + (15 if pop == 'EUR' else 0), 
                'depth': float(depth[i] + rng.normal(0, 3)),
                'pop': pop,
            })
qdf = pd.DataFrame(quant_rows)            

In [4]:
qdf.head()

,chrom,pos,fst,rate,eig,ci_lo,ci_hi,depth,pop
0,chr1,1090224,0.044858,5.824997,0.811761,-0.455446,10.059748,29.708953,AFR
1,chr1,1217540,0.088215,5.465193,0.749436,-0.165021,10.434385,31.654820,AFR
2,chr1,1321229,0.038109,5.644535,0.775922,0.183732,10.394984,26.824892,AFR
3,chr1,1351792,0.063331,5.054980,0.663487,0.311840,10.397019,32.699465,AFR
4,chr1,1620963,0.126832,5.321591,0.598848,0.114335,11.571794,19.297649,AFR


In [5]:
viewer1 = (
    Tracks(chrom_sizes)
    .add_point_track(
        qdf, 'Scatter', x='pos', y='fst', group_by='pop', 
        tip_fmt=False,
    )
    .add_line_track(
        qdf, 'Lines', x='pos', y='rate', group_by='pop',
        step='post', tip_fmt=False, height=60,
    )
    .add_fill_track(
        qdf, 'Fill btwn', x='pos', y_hi='ci_hi', y_lo='ci_lo', 
        group_by='pop', height=60, tip_fmt=False,
    )
    .add_fill_track(
        qdf[qdf['pop'] == 'AFR'], 'Fill baseln', x='pos', y='eig',
        step='post', height=60, tip_fmt=False,
    )
    .add_histogram_track(
        qdf, 'Histogram', x='pos', y='depth', group_by='pop',
        stack=True,
        height=60, tip_fmt=False, 
    )
    .add_gene_track(assembly='hg38', height = 100)
)
viewer1

## Log-scaled y-axis

Point and line tracks take `y_scale='log'` to draw their y-axis on a base-10 log scale, ticked at powers of ten. This is set per track, so log and linear tracks can sit in the same viewer. Rows with values ≤ 0 have no position on a log axis; they are dropped with a warning. A `y_range` is given in data units, e.g. `y_range=(1e-8, 1)`, and tooltips show data values.

When zoomed out, values in a bin are first aggregated (`aggregate='mean'`, `'sum'` or `'max'`) and the result is then placed on the log axis, so `'mean'` is the ordinary arithmetic mean.

In [6]:
qdf['pval'] = 10 ** -np.random.default_rng(1).exponential(1.5, len(qdf))

viewer_log = (
    Tracks(chrom_sizes)
    .add_point_track(
        qdf, 'p-value', x='pos', y='pval', group_by='pop',
        y_scale='log', tip_fmt='{group}: {value:.2g}',
    )
    .add_line_track(
        qdf, 'Rate (linear)', x='pos', y='rate', group_by='pop', step='post',
    )
)
viewer_log

## Gaps in line and fill tracks

A missing value (`NaN`) breaks a line or fill track, as in matplotlib: the line or band stops at the last sample before the missing value and resumes at the next sample after it. For a fill, a `NaN` in `y`, or in either `y_lo` or `y_hi`, makes a gap. Positions that simply have no row are bridged, so add rows with `NaN` where the track should break. When zoomed out, a bin is only a gap if all its samples are `NaN`; other bins aggregate their non-missing samples.

In [7]:
qgap = qdf.copy()
missing = (qgap['chrom'] == 'chr1') & qgap['pos'].between(60_000_000, 120_000_000)
qgap.loc[missing, ['rate', 'ci_lo', 'ci_hi']] = np.nan

viewer_gap = (
    Tracks(chrom_sizes)
    .add_line_track(
        qgap, 'Lines', x='pos', y='rate', group_by='pop',
        step='post', tip_fmt=False,
    )
    .add_fill_track(
        qgap, 'Fill btwn', x='pos', y_hi='ci_hi', y_lo='ci_lo',
        group_by='pop', tip_fmt=False,
    )
)
viewer_gap

## Hiding, showing and selecting tracks

`remove_track` deletes a track and its data. To take a track out of view and bring it back later without re-rendering it, use `hide_track` and `show_track` instead: a hidden track keeps its data and its place, so showing it again is instant. All three accept a track name, which selects every track with that name, or a position in the track list (negative positions count from the end).

In [8]:
viewer6 = (
    Tracks(chrom_sizes)
    .add_point_track(qdf, 'Scatter', x='pos', y='fst', group_by='pop', tip_fmt=False)
    .add_line_track(qdf, 'Lines', x='pos', y='rate', group_by='pop', tip_fmt=False)
    .add_fill_track(
        qdf, 'Fill btwn', x='pos', y_hi='ci_hi', y_lo='ci_lo',
        group_by='pop', tip_fmt=False,
    )
)
viewer6

`added_tracks` lists the names of all tracks, hidden ones included, in display order:

In [9]:
viewer6.added_tracks()

['Scatter', 'Lines', 'Fill btwn']

Hide a track and show it again:

In [10]:
viewer6.hide_track('Scatter')
viewer6.show_track('Scatter');

`select_tracks` shows only the listed tracks, in the order listed, and hides the rest. Give the names as separate arguments or as one list. Here it shows the fill track above the line track and hides the scatter track. Passing `viewer6.added_tracks()` shows everything again.

In [11]:
viewer6.select_tracks('Fill btwn', 'Lines');

## Highlighting

In [12]:
viewer2 = (
    Tracks(chrom_sizes)
    .add_gene_track(assembly='hg38',
                    height = 100, 
                    highlight=['LRRC7', 'LRRC40', 'CTH', 'SRSF11', 'CTH'], 
                    highlight_color="#ee55c5",
                    )
)
# zoom to region interest for demonstration purposes
viewer2.zoom_to('chr1', center=70000000, window=1200000)

In [13]:
viewer3 = (
    Tracks(chrom_sizes)
    .add_gene_track(assembly='hg38',
                    height = 100, 
                    highlight={                                                                   
                        'fill':      ['ANKRD13C', 'LRRC40', 'CTH'],                                           
                        'stroke':    ['LRRC40'],                                                     
                        'outline':   ['BRCA1'],                                                   
                        'color':     ['LRRC40'],                                                    
                        'bold':      ['LRRC7', 'LRRC40'],                                           
                        'italic':    ['ANKRD13C'],                                                     
                        'underline': ['CTH'],                                                    
                        'halo':      ['SRSF11'],                                                   
                    }, 
                    highlight_fill_color    = "#e03a4e",                  
                    highlight_spine_color   = "#d80ce6",                                          
                    highlight_outline_color = "#000000",                                          
                    highlight_label_color   = "#169f4a",                                          
                    highlight_halo_color    = "#79CAD32E",    
                    )
)
# zoom to region interest for demonstration purposes
viewer3.zoom_to('chr1', center=70300000, window=600000)

## Segments

Synthetic segment data:

In [14]:
#| echo: false
df_list = []
for chrom, csz in chrom_sizes.items():
    for group in ['EUR', 'EAS']:
        for ind in [f'{group}_ind{i}' for i in range(1, 11)]:
            for kind in ['Neanderthal', 'Denisovan']:
                n_points = 100
                positions = np.sort(rng.integers(0, csz, n_points))
                length = rng.integers(10000, 1000000, n_points)
                df = pd.DataFrame({
                    'chrom': chrom,
                    'start': positions,
                    'end': positions + length,
                    'ind': ind,
                    'reg': group,
                    'ancestry': kind,
                })
                df_list.append(df)
segs = pd.concat(df_list, ignore_index=True)

hm_df = segs[['chrom', 'start', 'end', 'ind', 'reg']].rename(
    columns={'ind': 'sample', 'reg': 'pop'}
)

In [15]:
segs.head()

,chrom,start,end,ind,reg,ancestry
0,chr1,9264851,9642214,EUR_ind1,EUR,Neanderthal
1,chr1,14200255,15198134,EUR_ind1,EUR,Neanderthal
2,chr1,15108496,15182094,EUR_ind1,EUR,Neanderthal
3,chr1,15970719,16967259,EUR_ind1,EUR,Neanderthal
4,chr1,22242645,22531349,EUR_ind1,EUR,Neanderthal


In [16]:
viewer4 = (
    Tracks(chrom_sizes)
    .add_segment_track(
        segs.loc[segs['reg'] == 'EUR'], 'EUR', group_by='ancestry', individual_col='ind',
        tip_fmt=False, stack=True,
        height=60, zoom_windows=(512, 2048, 8192),
    )   
    .add_segment_track(
        segs.loc[segs['reg'] == 'EAS'], 'EAS', group_by='ancestry', individual_col='ind',
          tip_fmt=False, stack=True,
        height=60, zoom_windows=(512, 2048, 8192),
    )    
    .add_heatmap_track(
        segs, 'Individuals',
        individual_col='ind',
            group_by='reg',
        height=90,   # will be expanded to accommodate on px per segment row
        zoom_windows=2500, # max is 8192 or 16384 depending on GPU
        tip_fmt=False,
    )    
    .add_gene_track(assembly='hg38', height = 100)
)
viewer4

## Tooltip

Each track contributes to the cursor tooltip unless `tip_fmt=False`. Use `tip_fmt` to control what each track contributes to the tooltip. Format placeholders use Python-style `{key}` or `{key:.Nf}` syntax.

Available keys per track type:
- **Gene**: `name`, `strand`, `start`, `end`
- **Scatter / Line**: `group`, `value`, `x`
- **Fill**: `group`, `lo`, `hi`, `x`
- **Histogram**: `group`, `value`, `x`
- **Segment**: `group`
- **Heatmap**: `group`, `individual`, `nInd`

If a format string references an invalid key, the tooltip shows the available keys as a hint.

In [17]:
viewer5 = (
    Tracks(chrom_sizes)
    .add_point_track(
        qdf, 'Scatter', x='pos', y='fst', group_by='pop',
        tip_fmt="{value:.4f} is the value for {group}",
    )
    .add_line_track(
        qdf, 'Lines', x='pos', y='rate', group_by='pop',
        tip_fmt="For {group} the value is {value:.2f}",
    )
    .add_gene_track(
        assembly='hg38', height=44, 
        tip_fmt="The gene at this pos is {name}",
    )
)
viewer5

## Theme

The widget has a light and dark default theme determined by the vscodenb package if installed. You get the active theme defaults like this:

In [18]:
from geneinfo.widget import get_default_theme, set_default_theme
theme = get_default_theme()
theme

{'bg': '#ffffffff',
 'fg': '#1a1a2eff',
 'panel': '#ffffffff',
 'border': '#ffffffff',
 'input_bg': '#ffffffff',
 'input_fg': '#1a1a2eff',
 'input_border': '#c0c0ccff',
 'focus_border': '#3355ddff',
 'axis_text': '#888899ff',
 'track_label': '#555566ff',
 'gene_exon': '#4488ccff',
 'gene_spine': '#666688ff',
 'gene_label': '#000000ff',
 'highlight_fill': '#e03a4eff',
 'highlight_spine': '#d80ce6ff',
 'highlight_outline': '#000000ff',
 'highlight_label': '#169f4aff',
 'highlight_halo': '#79cad32e',
 'sidebar_w': 96}

You can change the theme colors and save them back for use in subsequent viewers:

In [19]:
# theme['input_bg'] = 'pink'
set_default_theme(theme)